In [ ]:
!pip install -q timm fvcore

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import timm
import pandas as pd
from fvcore.nn import FlopCountAnalysis

# 1. Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

assert torch.cuda.is_available(), "Switch your Colab runtime to T4 GPU first!"
device_cpu = torch.device("cpu")
device_gpu = torch.device("cuda")

# 2. Load Held-Out Dataset (Imagenette Validation Set)
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = torchvision.datasets.Imagenette(
    root="./data", split="val", size="320px", download=True, transform=transform
)
sample_img, sample_label = dataset[0]
img_input = sample_img.unsqueeze(0)  # Shape: (1, 3, 256, 256)
print(f"Loaded Imagenette val set: {len(dataset)} images | Sample shape: {tuple(img_input.shape)}\n")

# 3. Define Models
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=128, hidden_size=256, num_layers=2, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

seq_input = torch.randn(1, 128, 128)  # (Batch=1, Seq_Len=128, Features=128)

models_dict = {
    "MobileViT-S": (timm.create_model("mobilevit_s", pretrained=True).eval(), img_input),
    "MobileNetV2": (timm.create_model("mobilenetv2_100", pretrained=True).eval(), img_input),
    "LSTM (2-Layer)": (LSTMClassifier().eval(), seq_input),
}

# 4. Profile Params, FLOPs, and Validate CPU vs. GPU Outputs
results = []

for name, (model, sample_in) in models_dict.items():
    # Parameter count
    params = sum(p.numel() for p in model.parameters())

    # FLOPs count (fvcore counts MACs; 1 MAC = 2 FLOPs)
    model_cpu = model.to(device_cpu)
    flops_analyzer = FlopCountAnalysis(model_cpu, sample_in)
    flops_analyzer.unsupported_ops_warnings(False)
    macs = flops_analyzer.total()
    flops = macs * 2

    # CPU vs. GPU Output Validation (atol=1e-3)
    with torch.no_grad():
        out_cpu = model_cpu(sample_in)
        model_gpu = model.to(device_gpu)
        out_gpu = model_gpu(sample_in.to(device_gpu)).cpu()

    max_diff = (out_cpu - out_gpu).abs().max().item()
    passed = torch.allclose(out_cpu, out_gpu, atol=1e-3)

    results.append({
        "Model": name,
        "Input_Shape": str(tuple(sample_in.shape)),
        "Params_M": round(params / 1e6, 3),
        "MACs_G": round(macs / 1e9, 4),
        "FLOPs_G": round(flops / 1e9, 4),
        "Max_Abs_Diff": f"{max_diff:.2e}",
        "Match_atol_1e-3": passed,
    })

    # Move back to CPU to keep GPU memory clean
    model.to(device_cpu)
    torch.cuda.empty_cache()

specs_df = pd.DataFrame(results)
specs_df.to_csv("model_specs.csv", index=False)
print(specs_df.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
100%|██████████| 342M/342M [00:08<00:00, 42.5MB/s]


Loaded Imagenette val set: 3925 images | Sample shape: (1, 3, 256, 256)



model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

stages.2.1.transformer.0.attn.attn_drop, stages.2.1.transformer.1.attn.attn_drop, stages.3.1.transformer.0.attn.attn_drop, stages.3.1.transformer.1.attn.attn_drop, stages.3.1.transformer.2.attn.attn_drop, stages.3.1.transformer.3.attn.attn_drop, stages.4.1.transformer.0.attn.attn_drop, stages.4.1.transformer.1.attn.attn_drop, stages.4.1.transformer.2.attn.attn_drop


         Model      Input_Shape  Params_M  MACs_G  FLOPs_G Max_Abs_Diff  Match_atol_1e-3
   MobileViT-S (1, 3, 256, 256)     5.579  1.8558   3.7116     1.22e-03             True
   MobileNetV2 (1, 3, 256, 256)     3.505  0.4100   0.8200     4.98e-06             True
LSTM (2-Layer)    (1, 128, 128)     0.924  0.0000   0.0000     7.45e-09             True


In [ ]:
# Analytical MACs/FLOPs for 2-layer LSTM (T=128, I=128, H=256, C=10)
T, I, H, C = 128, 128, 256, 10
lstm_macs = T * 4 * (I + H) * H + T * 4 * (H + H) * H + H * C
lstm_flops = lstm_macs * 2

specs_df.loc[specs_df["Model"] == "LSTM (2-Layer)", "MACs_G"] = round(lstm_macs / 1e9, 4)
specs_df.loc[specs_df["Model"] == "LSTM (2-Layer)", "FLOPs_G"] = round(lstm_flops / 1e9, 4)

specs_df.to_csv("model_specs.csv", index=False)
print(specs_df.to_string(index=False))

         Model      Input_Shape  Params_M  MACs_G  FLOPs_G Max_Abs_Diff  Match_atol_1e-3
   MobileViT-S (1, 3, 256, 256)     5.579  1.8558   3.7116     1.22e-03             True
   MobileNetV2 (1, 3, 256, 256)     3.505  0.4100   0.8200     4.98e-06             True
LSTM (2-Layer)    (1, 128, 128)     0.924  0.1174   0.2349     7.45e-09             True


In [ ]:
import time
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Re-use models if already loaded, otherwise create them
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=128, hidden_size=256, num_layers=2, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model_builders = {
    "MobileViT-S": lambda: timm.create_model("mobilevit_s", pretrained=True).eval(),
    "MobileNetV2": lambda: timm.create_model("mobilenetv2_100", pretrained=True).eval(),
    "LSTM (2-Layer)": lambda: LSTMClassifier().eval(),
}

batch_sizes = [1, 2, 4, 8, 16, 32]
configs = [
    ("CPU", "FP32"),
    ("GPU", "FP32"),
    ("GPU", "FP16"),
]

def make_batch(model_name, batch_size):
    if model_name == "LSTM (2-Layer)":
        return torch.randn(batch_size, 128, 128)
    # Grab real preprocessed Imagenette tensors when available
    if "dataset" in globals():
        imgs = [dataset[i % len(dataset)][0] for i in range(batch_size)]
        return torch.stack(imgs, dim=0)
    return torch.randn(batch_size, 3, 256, 256)

def benchmark_inference(model, cpu_batch, device_type, precision, batch_size):
    device = torch.device("cuda" if device_type == "GPU" else "cpu")
    use_fp16 = (device_type == "GPU" and precision == "FP16")

    # Keep CPU large batches fast enough for Colab (~2-3 min total)
    if device_type == "CPU":
        warmups = 2
        repeats = 10 if batch_size <= 8 else 5
    else:
        warmups = 5
        repeats = 20

    model = model.to(device)
    if device_type == "GPU":
        torch.cuda.reset_peak_memory_stats()

    dev_batch = cpu_batch.to(device)

    # 1. Warm-up
    with torch.no_grad():
        for _ in range(warmups):
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                _ = model(dev_batch)
        if device_type == "GPU":
            torch.cuda.synchronize()

    # 2. Steady-State Latency (compute only, input already on device)
    steady_times = []
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.perf_counter()
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                _ = model(dev_batch)
            if device_type == "GPU":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            steady_times.append((t1 - t0) * 1000.0)

    # 3. End-to-End Latency (H2D transfer + compute + D2H copy back to CPU)
    e2e_times = []
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.perf_counter()
            x = cpu_batch.to(device)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                y = model(x)
            _ = y.cpu()
            if device_type == "GPU":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            e2e_times.append((t1 - t0) * 1000.0)

    peak_mem_mb = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if device_type == "GPU" else np.nan
    )

    med_steady = float(np.median(steady_times))
    p95_steady = float(np.percentile(steady_times, 95))
    med_e2e = float(np.median(e2e_times))
    throughput = batch_size / (med_steady / 1000.0)

    return med_steady, p95_steady, med_e2e, throughput, peak_mem_mb

rows = []

for model_name, builder in model_builders.items():
    print(f"\n=== Benchmarking {model_name} ===")
    for device_type, precision in configs:
        model = builder()
        for bs in batch_sizes:
            try:
                cpu_batch = make_batch(model_name, bs)
                med_s, p95_s, med_e2e, tput, mem_mb = benchmark_inference(
                    model, cpu_batch, device_type, precision, bs
                )
                rows.append({
                    "Model": model_name,
                    "Device": device_type,
                    "Precision": precision,
                    "Batch_Size": bs,
                    "Steady_Median_ms": round(med_s, 3),
                    "Steady_P95_ms": round(p95_s, 3),
                    "E2E_Median_ms": round(med_e2e, 3),
                    "Throughput_samples_s": round(tput, 2),
                    "Peak_GPU_Mem_MB": round(mem_mb, 2) if not np.isnan(mem_mb) else np.nan,
                    "Status": "OK",
                })
                print(
                    f"{device_type:3} {precision:4} | BS={bs:2} | "
                    f"Steady: {med_s:7.2f} ms | E2E: {med_e2e:7.2f} ms | "
                    f"Tput: {tput:8.1f} img/s"
                )
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if "out of memory" in str(e).lower():
                    rows.append({
                        "Model": model_name, "Device": device_type, "Precision": precision,
                        "Batch_Size": bs, "Steady_Median_ms": np.nan, "Steady_P95_ms": np.nan,
                        "E2E_Median_ms": np.nan, "Throughput_samples_s": np.nan,
                        "Peak_GPU_Mem_MB": np.nan, "Status": "OOM",
                    })
                    print(f"{device_type:3} {precision:4} | BS={bs:2} | OOM!")
                else:
                    raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

inf_df = pd.DataFrame(rows)
inf_df.to_csv("day5_inference_cpu_gpu.csv", index=False)
print("\nSaved day5_inference_cpu_gpu.csv")


=== Benchmarking MobileViT-S ===
CPU FP32 | BS= 1 | Steady:  198.36 ms | E2E:  164.02 ms | Tput:      5.0 img/s
CPU FP32 | BS= 2 | Steady:  177.26 ms | E2E:  170.67 ms | Tput:     11.3 img/s
CPU FP32 | BS= 4 | Steady:  598.42 ms | E2E:  500.98 ms | Tput:      6.7 img/s
CPU FP32 | BS= 8 | Steady: 1085.40 ms | E2E: 1051.48 ms | Tput:      7.4 img/s
CPU FP32 | BS=16 | Steady: 3013.34 ms | E2E: 2721.36 ms | Tput:      5.3 img/s
CPU FP32 | BS=32 | Steady: 5978.73 ms | E2E: 6531.35 ms | Tput:      5.4 img/s
GPU FP32 | BS= 1 | Steady:    9.35 ms | E2E:    8.99 ms | Tput:    107.0 img/s
GPU FP32 | BS= 2 | Steady:    8.00 ms | E2E:    8.48 ms | Tput:    250.0 img/s
GPU FP32 | BS= 4 | Steady:   12.42 ms | E2E:   12.80 ms | Tput:    322.0 img/s
GPU FP32 | BS= 8 | Steady:   23.41 ms | E2E:   24.49 ms | Tput:    341.7 img/s
GPU FP32 | BS=16 | Steady:   44.80 ms | E2E:   46.48 ms | Tput:    357.1 img/s
GPU FP32 | BS=32 | Steady:   88.00 ms | E2E:   92.49 ms | Tput:    363.6 img/s
GPU FP16 | BS= 1 |

In [ ]:
from torch.utils.data import DataLoader, Subset
import torchvision
from timm.data import resolve_model_data_config, create_transform

IMAGENETTE_TO_IMAGENET = torch.tensor([0, 217, 482, 491, 497, 566, 569, 571, 574, 701])

# Fixed-seed random 500-image sample covering all 10 classes
g = torch.Generator().manual_seed(42)
eval_indices = torch.randperm(len(dataset), generator=g)[:500].tolist()

acc_rows = []
device = torch.device("cuda")

for model_name in ["MobileViT-S", "MobileNetV2"]:
    model = model_builders[model_name]().to(device)

    # Pull each model's exact native preprocessing (mean, std, resize, crop)
    data_cfg = resolve_model_data_config(model)
    native_tf = create_transform(**data_cfg, is_training=False)

    eval_ds = torchvision.datasets.Imagenette(
        root="./data", split="val", size="320px", download=False, transform=native_tf
    )
    eval_loader = DataLoader(Subset(eval_ds, eval_indices), batch_size=32, shuffle=False)

    correct_fp32 = 0
    correct_fp16 = 0
    agree = 0
    total = 0
    max_logit_diff = 0.0

    with torch.no_grad():
        for imgs, labels in eval_loader:
            imgs = imgs.to(device)
            target_1k = IMAGENETTE_TO_IMAGENET[labels].to(device)

            out_32 = model(imgs)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out_16 = model(imgs)

            pred_32 = out_32.argmax(dim=1)
            pred_16 = out_16.argmax(dim=1)

            correct_fp32 += (pred_32 == target_1k).sum().item()
            correct_fp16 += (pred_16 == target_1k).sum().item()
            agree += (pred_32 == pred_16).sum().item()
            total += labels.size(0)

            diff = (out_32 - out_16.float()).abs().max().item()
            max_logit_diff = max(max_logit_diff, diff)

    acc_rows.append({
        "Model": model_name,
        "Eval_Samples": total,
        "FP32_Top1_Acc_%": round(100.0 * correct_fp32 / total, 2),
        "FP16_Top1_Acc_%": round(100.0 * correct_fp16 / total, 2),
        "FP32_FP16_Agreement_%": round(100.0 * agree / total, 2),
        "Max_Logit_Diff": round(max_logit_diff, 4),
    })

    del model
    torch.cuda.empty_cache()

acc_df = pd.DataFrame(acc_rows)
acc_df.to_csv("day5_precision_accuracy.csv", index=False)
print(acc_df.to_string(index=False))

      Model  Eval_Samples  FP32_Top1_Acc_%  FP16_Top1_Acc_%  FP32_FP16_Agreement_%  Max_Logit_Diff
MobileViT-S           500             85.0             85.0                  100.0          0.0662
MobileNetV2           500             78.2             78.4                   99.8          0.0497
